# オーケストレーター ローカルLLM再蒸留ノートブック

**ベースモデル: `llm-jp/llm-jp-3.1-1.8b-instruct4`**
(2026-08-19更新: `llm-jp-3-1.8b-instruct3` から乗り換え。同じ1.8Bサイズのまま
日本語MT-Benchが4.64→6.30に向上した新世代モデル)

`distill_claude_authored.jsonl` でLoRAファインチューニングし、
マージ後にGGUF(Q4_K_M)へ変換するところまでを行います。

**実行前に**: 上部メニュー → ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータを **GPU (T4)** にしてください。

## ロールバックについて
出力ファイル名を `llm-jp-3.1-1.8b-instruct4-Q4_K_M.gguf` と、
旧モデル(`llm-jp-3-1.8b-instruct3-Q4_K_M.gguf`)とは別名にしています。
Mac側では旧ファイルを**上書きせず、新ファイルとして並べて配置**するので、
うまくいかなければ `orchestrator_v4.py` の `LOCAL_MODEL_PATH` を
旧ファイル名に戻すだけで即座に元に戻せます(詳細は `DISTILLATION_GUIDE.md` 参照)。

## 手順
1. 環境セットアップ
2. 学習データのアップロード(`distill_claude_authored.jsonl`)
3. ベースモデル読み込み
4. データセット整形
5. LoRA設定・学習
6. マージ
7. GGUF変換・量子化(Q4_K_M)
8. ダウンロード


## 1. 環境セットアップ

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes sentencepiece datasets "torchao>=0.16.0"


## 2. 学習データのアップロード

`distill_claude_authored.jsonl` をローカルからアップロードしてください
(GitHubリポジトリの最新版を `git pull` してから対象ファイルを選んでください)。


In [ ]:
from google.colab import files
uploaded = files.upload()  # distill_claude_authored.jsonl を選択
DATA_PATH = list(uploaded.keys())[0]
print("読み込んだファイル:", DATA_PATH)


In [ ]:
import json
examples = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        examples.append(json.loads(line))
print(f"件数: {len(examples)}")
print(examples[0])


## 3. ベースモデル読み込み

`llm-jp-3.1-1.8b-instruct4`(2025年5月公開)。`llm-jp-3-1.8b` の継続事前学習+
事後学習改善版で、同サイズのまま指示追従性が大きく向上しています。
アーキテクチャは引き続きLlama系(`LlamaForCausalLM`)のため、
LoRAの対象モジュール名はそのまま流用できます。


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "llm-jp/llm-jp-3.1-1.8b-instruct4"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
print(model.config.architectures)


## 4. データセット整形

`orchestrator_v4.py` の `ask_local()` に実際に埋め込まれているsystemプロンプトと
**一言一句同じもの**を使い、チャットテンプレート(`apply_chat_template`)で整形します。
(推論時にllama.cppが `--jinja` フラグで同じテンプレートを使うため、学習時も揃えます)

※ 今後 `ask_local()` 側のsystemプロンプトを変更した場合は、このセルも
必ず追従して更新してください(プロンプトと学習データの方針がズレると効果が薄れます)。


In [ ]:
# orchestrator_v4.py の ask_local() 内、system変数と完全一致させること(2026-08-19時点)
SYSTEM_PROMPT = """あなたは日本語AIアシスタントです。

通常の質問には自然な日本語で回答してください。

正確性を重視し、必要なら手順や理由も説明してください。

Pythonコード修正は、ユーザーが明示的に依頼した場合のみ行ってください。

【回答スタイルの指針】
- 結論を先に、理由は簡潔に。前置きは最小限に。
- 断定できないことは断定しない。推測は「〜の可能性があります」と明示する。
- 相手の設計判断を否定せず、まず理由を尋ねてから助言する。

【ログ診断の考え方(例)】
ログに例外が出ていても、try/exceptで捕捉済み・処理が継続していて実害がないものは「ノイズ」として優先度を下げる。
例: 「'NeWtOnS' is not defined」のようなエラーでも、直後に処理が正常継続していればクラッシュではないため緊急対応は不要、と判断する。
重大度が高いのは「処理が停止した」「同じエラーが短時間に大量発生した」「認証・課金・データ破損に関わる」場合。

【修正指示の書き方(例)】
悪い例:「バグを直して」→ 対象箇所も原因も不明で誤修正のリスクが高い。
良い例:「agent_log_doctor.pyのextract_unclassified_patterns関数で、ログが空のときにIndexErrorが出るので、空リストの場合は早期returnするようにして」→ 対象ファイル・関数・症状・望む挙動が明確。
"""

def format_example(ex):
    chat = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ex["instruction"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    return tokenizer.apply_chat_template(chat, tokenize=False)

texts = [format_example(ex) for ex in examples]
print(texts[0][:500])


In [ ]:
from datasets import Dataset

MAX_LEN = 1024

def tokenize_fn(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )
    out["labels"] = [ids.copy() for ids in out["input_ids"]]
    return out

ds = Dataset.from_dict({"text": texts})
ds = ds.map(tokenize_fn, batched=True, remove_columns=["text"])
ds.set_format(type="torch")
print(ds)


## 5. LoRA設定・学習

件数が54件と少ないので、epoch数は3〜5程度から様子見してください
(増やしすぎると過学習で応答が不自然になります)。


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

EPOCHS = 3

training_args = TrainingArguments(
    output_dir="/content/lora_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS,
    learning_rate=1e-4,
    logging_steps=5,
    save_strategy="no",
    bf16=True,
    report_to=[],
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    data_collator=data_collator,
)

trainer.train()


## 6. マージ(LoRAをベースモデルに統合)

In [ ]:
merged_model = model.merge_and_unload()

MERGED_DIR = "/content/merged_model"
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("マージ済みモデルを保存しました:", MERGED_DIR)


## 7. GGUF変換・量子化(Q4_K_M)

llama.cppをビルドし、HF形式 → GGUF(f16) → Q4_K_M量子化、の順で変換します。
出力ファイル名は旧モデルと衝突しないよう `llm-jp-3.1-1.8b-instruct4-*` にしています。


In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt


In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile /content/llm-jp-3.1-1.8b-instruct4-f16.gguf --outtype f16


In [ ]:
# llama-quantize バイナリをビルド
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=OFF
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4


In [ ]:
!/content/llama.cpp/build/bin/llama-quantize /content/llm-jp-3.1-1.8b-instruct4-f16.gguf /content/llm-jp-3.1-1.8b-instruct4-Q4_K_M.gguf Q4_K_M


## 8. ダウンロード

生成された `llm-jp-3.1-1.8b-instruct4-Q4_K_M.gguf` をダウンロードし、
**既存の `llm-jp-3-1.8b-instruct3-Q4_K_M.gguf` は削除せずそのまま残した状態で**、
同じ `~/ai-orchestrator/llama.cpp/models/` ディレクトリに配置してください
(`DISTILLATION_GUIDE.md` の⑥デプロイの手順を参照)。


In [ ]:
from google.colab import files
files.download("/content/llm-jp-3.1-1.8b-instruct4-Q4_K_M.gguf")
